# Core Web Vitals Analysis

Ad-hoc exploration on top of the curated Parquet tables produced by `make run`.
Run the pipeline first (`make seed && make run`) before using this notebook.

This is optional/exploratory -- the pipeline itself doesn't depend on it.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.utils.spark import get_spark
from src.utils.paths import as_posix
from src.config.settings import PATHS

spark = get_spark("web-vitals-notebook")

daily = spark.read.parquet(as_posix(PATHS.curated / "core_web_vitals_daily"))
rankings = spark.read.parquet(as_posix(PATHS.curated / "route_performance_rankings"))
before_after = spark.read.parquet(as_posix(PATHS.curated / "before_after_comparisons"))
regressions = spark.read.parquet(as_posix(PATHS.curated / "regression_events"))
device_breakdowns = spark.read.parquet(as_posix(PATHS.curated / "device_breakdowns"))

daily.printSchema()

## Health score trend over time, by device type

In [ ]:
from pyspark.sql import functions as F

trend = (
    daily.groupBy("date", "device_type")
    .agg(F.round(F.avg("overall_health_score"), 1).alias("avg_health_score"))
    .orderBy("date", "device_type")
)
trend.show(30)

In [ ]:
# Optional: plot if pandas + matplotlib are available (not required to run the pipeline itself).
try:
    import matplotlib.pyplot as plt

    pdf = trend.toPandas().pivot(index="date", columns="device_type", values="avg_health_score")
    pdf.plot(figsize=(10, 5), title="Overall Health Score by Device Type")
    plt.ylabel("Health Score (0-100)")
    plt.show()
except ImportError:
    print("Install pandas + matplotlib to render this chart: pip install pandas matplotlib")

## Worst routes right now (most recent date)

In [ ]:
latest_date = rankings.agg(F.max("date")).collect()[0][0]
(
    rankings.where(F.col("date") == latest_date)
    .orderBy(F.col("rank_overall").desc())
    .select("route", "device_type", "p75_lcp_ms", "p75_cls", "p75_inp_ms", "performance_score", "risk_level")
    .show(10)
)

## Active regressions, most severe first

In [ ]:
severity_rank = (
    F.when(F.col("severity") == "critical", 4)
    .when(F.col("severity") == "high", 3)
    .when(F.col("severity") == "medium", 2)
    .otherwise(1)
)
(
    regressions.withColumn("_rank", severity_rank)
    .orderBy(F.col("_rank").desc(), F.col("delta_percent").desc())
    .select("route", "device_type", "metric", "release_version", "severity", "baseline_value", "current_value", "delta_percent")
    .show(20, truncate=False)
)

## Device-class breakdown for the worst route

In [ ]:
worst_route = (
    rankings.where(F.col("date") == latest_date)
    .orderBy(F.col("rank_overall").desc())
    .first()
    .route
)
print(f"Worst route as of {latest_date}: {worst_route}")
device_breakdowns.where(F.col("route") == worst_route).orderBy("device_type", "device_class").show(20)

In [ ]:
spark.stop()